# 🧬 RFdiffusion + ProteinMPNN — 단백질 설계 파이프라인

단백질 설계를 **집짓기**에 비유하면 이해가 쉽습니다.

- **RFdiffusion** = 건축가. "몇 층짜리 건물을 어떤 모양으로 지을까"만 정합니다. 즉, 아미노산이 3차원 공간에서 어떻게 접힐지 **뼈대(backbone) 좌표**만 설계하고, 어떤 아미노산인지는 아직 정하지 않습니다.
- **ProteinMPNN** = 인테리어 시공업자. 건축가가 그린 뼈대(설계도)를 보고, "이 뼈대가 안정적으로 유지되려면 어느 자리에 어떤 아미노산(벽돌)을 놓아야 하는가"를 계산해서 실제 서열(문자열)을 채워 넣습니다.

두 모델을 이어 붙이면 **"임의의 새로운 단백질 구조를 만들고, 그 구조를 실제로 접을 수 있는 아미노산 서열까지 뽑아내는"** 파이프라인이 완성됩니다.

## 진행 순서
1. GPU 런타임 확인
2. RFdiffusion + ProteinMPNN 설치 (가중치 다운로드 포함, 몇 분 소요)
3. RFdiffusion으로 무조건부(unconditional) 단일 사슬 백본 생성
4. 생성된 백본을 3D로 시각화
5. ProteinMPNN으로 그 백본에 맞는 서열 설계
6. 결과(PDB + FASTA) 정리 및 다운로드

> **주의**: 반드시 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU`로 설정한 뒤 진행하세요.

In [3]:
#@title 0. GPU 런타임 확인
!nvidia-smi

Mon Sep 21 06:41:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
#@title 0-1. 파이썬/토치 버전 확인 (설치 방식 결정용)
import sys
print("python:", sys.version)
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda, "| available:", torch.cuda.is_available())

python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
torch: 2.11.0+cu128 | cuda: 12.8 | available: True


## 1. 설치

GPU는 T4, 파이토치는 2.11(cu128), 파이썬은 3.13으로 확인됐습니다. RFdiffusion 원본 저장소는 훨씬 오래된 환경(파이썬 3.9~3.10, 구버전 torch+DGL)을 기준으로 만들어져서 최신 Colab 환경과 바로 안 맞을 가능성이 있습니다. 우선 설치를 시도하면서 에러가 나면 그 자리에서 맞춰가겠습니다.

먼저 **가중치(모델 파라미터) 다운로드는 백그라운드로 미리 돌려두고**, 그동안 코드 설치를 진행합니다. 가중치 파일이 커서(수 GB) 병렬로 받아야 시간을 아낄 수 있습니다.

In [5]:
#@title 1-1. 가중치 백그라운드 다운로드 시작
import os
os.makedirs("params", exist_ok=True)

get_ipython().system('apt-get install -y -qq aria2')

# RFdiffusion 기본(unconditional) 모델 가중치
dl_cmd = (
    "aria2c -q -x 16 -c "
    "https://files.ipd.uw.edu/pub/RFdiffusion/6f5902ac237024bdd0c176cb93063dc4/Base_ckpt.pt "
    "-d params -o Base_ckpt.pt > /content/dl_base.log 2>&1 &"
)
os.system(dl_cmd)
print("백그라운드 다운로드 시작함. 로그: /content/dl_base.log")

Selecting previously unselected package libcares2:amd64.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.8) ...
/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is

In [6]:
#@title 1-2. RFdiffusion 저장소 클론
get_ipython().system('git clone --quiet https://github.com/RosettaCommons/RFdiffusion.git')
get_ipython().system('ls RFdiffusion')

config	env	  helper_scripts  LICENSE    rfdiffusion  setup.py  tutorials
docker	examples  img		  README.md  scripts	  tests


In [7]:
#@title 1-3. 의존성 파일 확인
get_ipython().system('cat RFdiffusion/env/SE3Transformer/requirements.txt 2>/dev/null')
print('---setup.py---')
get_ipython().system('cat RFdiffusion/setup.py')
print('---env dir---')
get_ipython().system('ls RFdiffusion/env')

e3nn==0.3.3
wandb==0.12.0
pynvml==11.0.0
git+https://github.com/NVIDIA/dllogger#egg=dllogger
decorator==5.1.0
---setup.py---
from setuptools import setup, find_packages

setup(name='rfdiffusion',
      version='1.1.0',
      description='RFdiffusion is an open source method for protein structure generation.',
      author='Rosetta Commons',
      url='https://github.com/RosettaCommons/RFdiffusion',
      scripts=["scripts/run_inference.py"],
      packages=find_packages(),
      install_requires=['torch', 'se3-transformer'])
---env dir---
SE3nv.yml  SE3Transformer


In [8]:
#@title 1-4. SE3Transformer 서브패키지 확인
get_ipython().system('ls RFdiffusion/env/SE3Transformer')
print('---setup.py---')
get_ipython().system('cat RFdiffusion/env/SE3Transformer/setup.py 2>/dev/null')

Dockerfile  LICENSE  README.md	       scripts		setup.py
images	    NOTICE   requirements.txt  se3_transformer	tests
---setup.py---
from setuptools import setup, find_packages

setup(
    name='se3-transformer',
    packages=find_packages(),
    include_package_data=True,
    version='1.0.0',
    description='PyTorch + DGL implementation of SE(3)-Transformers',
    author='Alexandre Milesi',
    author_email='alexandrem@nvidia.com',
)


In [9]:
#@title 1-5. DGL 설치 시도 (현재 torch/cuda 조합에 맞는 빌드가 있는지 확인)
get_ipython().system('pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.5/cu124/repo.html 2>&1 | tail -30')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.4/409.4 kB 22.7 MB/s eta 0:00:00


In [10]:
#@title 1-6. dgl import 확인
import dgl
print(dgl.__version__)
import torch
g = dgl.graph(([0,1],[1,2]))
print(g)

ImportError: cannot import name 'Mapping' from 'collections' (/usr/lib/python3.13/collections/__init__.py)

In [11]:
#@title 1-7. RFdiffusion이 요구하는 공식 conda 환경 파일 확인
get_ipython().system('cat RFdiffusion/env/SE3nv.yml')

name: SE3nv
channels:
  - defaults
  - conda-forge
  - pytorch
  - dglteam
  - nvidia
dependencies:
  - python=3.9
  - pytorch=1.9
  - torchaudio
  - torchvision
  - cudatoolkit=11.1
  - dgl-cuda11.1
  - pip
  - pip:
    - hydra-core
    - pyrsistent


## 1-8. 방향 전환: 격리된 conda 환경 사용

방금 확인했듯 Colab 커널은 **파이썬 3.13 + torch 2.11**인데, RFdiffusion 공식 요구사항은 **파이썬 3.9 + torch 1.9 + CUDA 11.1 + 옛날 DGL**입니다. pip로 최신 DGL을 깔아도 실제로는 너무 오래된 대체 휠이 잡혀서 `collections.Mapping` 같은 파이썬 3.10+에서 제거된 문법 때문에 깨졌습니다.

비유하자면, 지금 커널은 "최신 스마트폰"인데 RFdiffusion은 "2021년형 구형 운영체제에서만 도는 앱"입니다. 앱을 억지로 최신 기기에 맞추는 대신, **기기 안에 구형 OS를 흉내내는 별도의 방(conda 가상환경)을 하나 더 만들어서** 그 안에서 앱을 돌리는 방식으로 갑니다. Colab 노트북 커널 자체를 바꾸는 게 아니라, `!` 셸 명령으로 그 가상환경의 파이썬 실행 파일을 직접 호출하는 방식입니다.

In [12]:
#@title 1-9. Miniconda 설치
import os
get_ipython().system('wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh')
get_ipython().system('bash /content/miniconda.sh -b -p /opt/miniconda3')
get_ipython().system('/opt/miniconda3/bin/conda --version')
get_ipython().system('/opt/miniconda3/bin/conda install -y -n base -c conda-forge mamba >/content/mamba_install.log 2>&1 &')
print("mamba 설치를 백그라운드로 시작함")

PREFIX=/opt/miniconda3
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /opt/miniconda3
conda 26.7.1
mamba 설치를 백그라운드로 시작함


In [14]:
#@title 1-11. conda 채널 ToS 동의 + mamba 설치 재시도
get_ipython().system('/opt/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main')
get_ipython().system('/opt/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r')
get_ipython().system('/opt/miniconda3/bin/conda install -y -n base -c conda-forge mamba > /content/mamba_install.log 2>&1')
get_ipython().system('tail -20 /content/mamba_install.log')

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
  reproc-cpp         pkgs/main::reproc-cpp-14.2.7-h7bdf020~ --> conda-forge::reproc-cpp-14.2.8.post0-h1c70be6_0 
  simdjson            pkgs/main::simdjson-3.10.1-hdb19cb5_0 --> conda-forge::simdjson-4.6.11-h4dbf13b_0 
  xz                         pkgs/main::xz-5.8.2-h448239c_0 --> conda-forge::xz-5.8.3-ha02ee65_1 
  zstd                     pkgs/main::zstd-1.5.7-h11fc155_0 --> conda-forge::zstd-1.5.7-hb78ec9c_7 

The following packages will be SUPERSEDED by a higher-priority channel:

  yaml-cpp             pkgs/main::yaml-cpp-0.9.0-he852c71_1 --> conda-forge::yaml-cpp-0.8.0-h54a6638_1 



Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done

Channel "defaults" has the following notices:
  [info] -- Thu Aug  6 00:00:00 2026
  main-x (Anaconda's new authenticated channel) is now generally

In [15]:
#@title 1-12. SE3nv 환경 생성 (백그라운드, 오래 걸릴 수 있음)
get_ipython().system("nohup /opt/miniconda3/bin/mamba env create -f RFdiffusion/env/SE3nv.yml > /content/env_create.log 2>&1 &")
print("SE3nv 환경 생성을 백그라운드로 시작함. env_create.log 를 폴링하세요.")

SE3nv 환경 생성을 백그라운드로 시작함. env_create.log 를 폴링하세요.


In [16]:
#@title 1-13. (대기하는 동안) ProteinMPNN 클론
get_ipython().system('git clone --quiet https://github.com/dauparas/ProteinMPNN.git')
get_ipython().system('ls ProteinMPNN')
get_ipython().system('ls ProteinMPNN/vanilla_model_weights | head')

ca_model_weights  LICENSE		 soluble_model_weights
colab_notebooks   outputs		 training
examples	  protein_mpnn_run.py	 vanilla_model_weights
helper_scripts	  protein_mpnn_utils.py
inputs		  README.md
v_48_002.pt
v_48_010.pt
v_48_020.pt
v_48_030.pt


In [17]:
#@title 1-14. SE3nv 환경 생성 진행 상황 확인
get_ipython().system('tail -30 /content/env_create.log')
get_ipython().system('ps aux | grep -i mamba | grep -v grep')

warning  libmamba 'repo.anaconda.com', a commercial channel hosted by Anaconda.com, is used.
    
warning  libmamba Please make sure you understand Anaconda Terms of Services.
    
warning  libmamba See: https://legal.anaconda.com/policies/en/
⚠ Shard Index for pkgs/main/linux-64 not available, falling back to flat repodata
Using Flat Repodata for pkgs/main/linux-64                                                      ✔ Starting
Using Flat Repodata for pkgs/main/linux-64                                                ✔ Done (1.9 sec)
⚠ Shard Index for pkgs/main/noarch not available, falling back to flat repodata
Using Flat Repodata for pkgs/main/noarch                                                        ✔ Starting
Using Flat Repodata for pkgs/main/noarch                                                  ✔ Done (0.1 sec)
⚠ Shard Index for pkgs/r/linux-64 not available, falling back to flat repodata
Using Flat Repodata for pkgs/r/linux-64                                                

In [23]:
#@title 1-15. SE3nv 환경 생성 완료 확인
get_ipython().system('ps aux | grep mamba | grep -v grep')
get_ipython().system('/opt/miniconda3/bin/conda env list')
get_ipython().system('/opt/miniconda3/envs/SE3nv/bin/python --version')


# conda environments:
#
# * -> active
# + -> frozen
base                     /opt/miniconda3
SE3nv                    /opt/miniconda3/envs/SE3nv

Python 3.9.25


In [24]:
#@title 1-16. SE3nv 환경에 RFdiffusion + SE3Transformer 설치
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
get_ipython().system(f'{PY} -m pip install -q -r RFdiffusion/env/SE3Transformer/requirements.txt 2>&1 | tail -30')

In [25]:
#@title 1-17. se3-transformer, rfdiffusion 설치
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
get_ipython().system(f'cd RFdiffusion/env/SE3Transformer && {PY} setup.py install 2>&1 | tail -40')

copying build/lib/se3_transformer/model/basis.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/model
creating /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/runtime
copying build/lib/se3_transformer/runtime/__init__.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/runtime
copying build/lib/se3_transformer/runtime/callbacks.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/runtime
copying build/lib/se3_transformer/runtime/metrics.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/runtime
copying build/lib/se3_transformer/runtime/inference.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/runtime
copying build/lib/se3_transformer/runtime/loggers.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-packages/se3_transformer/runtime
copying build/lib/se3_transformer/runtime/training.py -> /opt/miniconda3/envs/SE3nv/lib/python3.9/site-

In [26]:
#@title 1-18. rfdiffusion 설치
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
get_ipython().system(f'cd RFdiffusion && {PY} -m pip install -q -e . 2>&1 | tail -40')

In [27]:
#@title 1-19. 설치 검증 (import 체크)
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
check = '''
import torch, e3nn, se3_transformer, hydra, omegaconf
print("torch", torch.__version__, "cuda ok:", torch.cuda.is_available())
print("e3nn", e3nn.__version__)
import rfdiffusion
print("rfdiffusion import OK")
'''
with open("/content/check_env.py", "w") as f:
    f.write(check)
get_ipython().system(f'{PY} /content/check_env.py')

torch 1.9.1.post3 cuda ok: False
e3nn 0.3.3
rfdiffusion import OK


In [28]:
#@title 진단4: SE3nv의 torch가 GPU를 못 보는 이유
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
check = '''
import torch
print("version:", torch.__version__)
print("cuda compiled version:", torch.version.cuda)
print("is_available:", torch.cuda.is_available())
try:
    print(torch._C._cuda_getDeviceCount())
except Exception as e:
    print("err:", e)
'''
with open("/content/check2.py", "w") as f:
    f.write(check)
get_ipython().system(f'{PY} /content/check2.py')
get_ipython().system('nvidia-smi -L')
get_ipython().system('/opt/miniconda3/envs/SE3nv/bin/python -c "import torch; print(torch.backends.cudnn.version())"')

version: 1.9.1.post3
cuda compiled version: None
is_available: False
err: module 'torch._C' has no attribute '_cuda_getDeviceCount'
GPU 0: Tesla T4 (UUID: GPU-5091321d-8d0a-c4de-a253-f7b27a7d4e71)
None


In [29]:
#@title 1-20. torch를 CUDA 11.1 빌드로 재설치
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
get_ipython().system(f'{PY} -m pip uninstall -y torch torchvision torchaudio 2>&1 | tail -10')
get_ipython().system(f'{PY} -m pip install -q torch==1.9.1+cu111 torchvision==0.10.1+cu111 torchaudio==0.9.1 '
                      f'-f https://download.pytorch.org/whl/torch_stable.html 2>&1 | tail -30')

Found existing installation: torch 1.9.1.post3
Uninstalling torch-1.9.1.post3:
  Successfully uninstalled torch-1.9.1.post3
Found existing installation: torchvision 0.15.2a0
Uninstalling torchvision-0.15.2a0:
  Successfully uninstalled torchvision-0.15.2a0
Found existing installation: torchaudio 0.9.0a0+a85b239
Uninstalling torchaudio-0.9.0a0+a85b239:
  Successfully uninstalled torchaudio-0.9.0a0+a85b239


In [32]:
#@title 1-21. SE3nv torch GPU 인식 재확인
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
check = '''
import torch
print("torch:", torch.__version__, "| cuda build:", torch.version.cuda, "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
'''
with open("/content/check3.py", "w") as f:
    f.write(check)
get_ipython().system(f'{PY} /content/check3.py')

torch: 1.9.1+cu111 | cuda build: 11.1 | available: True
device: Tesla T4


In [33]:
#@title 1-22. 전체 임포트 재검증
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
check = '''
import torch, e3nn, dgl, hydra, omegaconf
print("torch", torch.__version__, torch.cuda.is_available())
print("dgl", dgl.__version__)
import se3_transformer
print("se3_transformer OK")
import rfdiffusion
print("rfdiffusion OK")
'''
with open("/content/check4.py", "w") as f:
    f.write(check)
get_ipython().system(f'{PY} /content/check4.py')

DGL backend not selected or invalid.  Assuming PyTorch for now.
Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)
torch 1.9.1+cu111 True
dgl 0.9.1post1
se3_transformer OK
rfdiffusion OK


## 2. 백본 생성 (RFdiffusion)

이제 실제로 "건축 설계도"를 그립니다. 아무 조건 없이(unconditional) **100개 아미노산 길이의 단일 사슬(monomer) 하나**를 새로 생성해봅니다. 확산 모델(diffusion model)이라 이름 그대로, 처음엔 완전히 무작위인 3차원 점들에서 시작해서 조금씩 "그럴듯한 단백질 모양"으로 노이즈를 제거해 나갑니다 (이미지 생성 모델이 노이즈에서 사진을 만들어내는 것과 같은 원리를, 3D 좌표에 적용한 것입니다).

In [34]:
#@title 2-1. RFdiffusion 실행 (SE3nv 환경의 파이썬으로 서브프로세스 호출)
PY = "/opt/miniconda3/envs/SE3nv/bin/python"
get_ipython().system('mkdir -p outputs')
get_ipython().system(f'{PY} RFdiffusion/scripts/run_inference.py \\\n  inference.output_prefix=outputs/design \\\n  inference.model_directory_path=params \\\n  inference.input_pdb=null \\\n  "contigmap.contigs=[100-100]" \\\n  inference.num_designs=1 \\\n  denoiser.noise_scale_ca=0 denoiser.noise_scale_frame=0')

[2026-09-21 06:56:44,306][__main__][INFO] - Found GPU with device_name Tesla T4. Will run RFdiffusion on Tesla T4
Reading models from params
[2026-09-21 06:56:44,307][rfdiffusion.inference.model_runners][INFO] - Reading checkpoint from params/Base_ckpt.pt
This is inf_conf.ckpt_path
params/Base_ckpt.pt
Assembling -model, -diffuser and -preprocess configs from checkpoint
USING MODEL CONFIG: self._conf[model][n_extra_block] = 4
USING MODEL CONFIG: self._conf[model][n_main_block] = 32
USING MODEL CONFIG: self._conf[model][n_ref_block] = 4
USING MODEL CONFIG: self._conf[model][d_msa] = 256
USING MODEL CONFIG: self._conf[model][d_msa_full] = 64
USING MODEL CONFIG: self._conf[model][d_pair] = 128
USING MODEL CONFIG: self._conf[model][d_templ] = 64
USING MODEL CONFIG: self._conf[model][n_head_msa] = 8
USING MODEL CONFIG: self._conf[model][n_head_pair] = 4
USING MODEL CONFIG: self._conf[model][n_head_templ] = 4
USING MODEL CONFIG: self._conf[model][d_hidden] = 32
USING MODEL CONFIG: self._conf[

In [35]:
#@title 2-2. 생성 결과 파일 확인
get_ipython().system('ls -la outputs/')
get_ipython().system('head -5 outputs/design_0.pdb')

total 68
drwxr-xr-x 4 root root  4096 Sep 21 06:58 .
drwxr-xr-x 1 root root  4096 Sep 21 06:56 ..
drwxr-xr-x 3 root root  4096 Sep 21 06:56 2026-09-21
-rw-r--r-- 1 root root 26800 Sep 21 06:58 design_0.pdb
-rw-r--r-- 1 root root 23208 Sep 21 06:58 design_0.trb
drwxr-xr-x 2 root root  4096 Sep 21 06:58 traj
ATOM      1  N   GLY A   1     -17.695   1.580  -1.029  1.00  0.00
ATOM      2  CA  GLY A   1     -16.526   0.730  -0.839  1.00  0.00
ATOM      3  C   GLY A   1     -15.314   1.288  -1.574  1.00  0.00
ATOM      4  O   GLY A   1     -14.193   1.242  -1.068  1.00  0.00
ATOM      5  N   GLY A   2     -15.551   1.751  -2.697  1.00  0.00


## 3. 생성된 백본 3D로 보기

py3Dmol로 방금 만든 뼈대 구조를 확인합니다. 아직 아미노산 종류는 정해지지 않았기 때문에(RFdiffusion은 좌표만 만듭니다), PDB 파일 안의 모든 잔기가 편의상 글리신(GLY)으로 표시되어 있을 수 있습니다 — 이건 정상입니다. 다음 단계에서 ProteinMPNN이 이 자리에 실제 아미노산을 채워 넣습니다.

In [36]:
#@title 3-1. 생성된 백본 시각화
get_ipython().system('pip install -q py3Dmol')
import py3Dmol
get_ipython().system('ls outputs/')

with open("outputs/design_0.pdb") as f:
    pdb_str = f.read()

view = py3Dmol.view(width=600, height=500)
view.addModel(pdb_str, "pdb")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

2026-09-21  design_0.pdb  design_0.trb	traj


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4. 서열 설계 (ProteinMPNN)

이제 "인테리어 시공업자" 차례입니다. ProteinMPNN은 순수 PyTorch로 짜여 있어서 DGL 같은 까다로운 의존성이 없습니다 — Colab 기본 커널(torch 2.11)에서 바로 돌립니다.

절차는 2단계입니다.
1. `parse_multiple_chains.py`: PDB 파일을 ProteinMPNN이 읽을 수 있는 jsonl 포맷으로 변환
2. `protein_mpnn_run.py`: 그 jsonl을 입력받아, 뼈대 좌표에 맞는 아미노산 서열을 여러 개(온도를 조절하며 샘플링) 뽑아냄

In [30]:
#@title 4-0. (미리) ProteinMPNN이 기본 커널에서 로드되는지만 확인
get_ipython().system('python -c "import torch; print(torch.__version__)"')
get_ipython().system('python ProteinMPNN/protein_mpnn_run.py --help | head -20')

2.11.0+cu128
usage: protein_mpnn_run.py [-h] [--suppress_print SUPPRESS_PRINT] [--ca_only]
                           [--path_to_model_weights PATH_TO_MODEL_WEIGHTS]
                           [--model_name MODEL_NAME] [--use_soluble_model]
                           [--seed SEED] [--save_score SAVE_SCORE]
                           [--save_probs SAVE_PROBS] [--score_only SCORE_ONLY]
                           [--path_to_fasta PATH_TO_FASTA]
                           [--conditional_probs_only CONDITIONAL_PROBS_ONLY]
                           [--conditional_probs_only_backbone CONDITIONAL_PROBS_ONLY_BACKBONE]
                           [--unconditional_probs_only UNCONDITIONAL_PROBS_ONLY]
                           [--backbone_noise BACKBONE_NOISE]
                           [--num_seq_per_target NUM_SEQ_PER_TARGET]
                           [--batch_size BATCH_SIZE] [--max_length MAX_LENGTH]
                           [--sampling_temp SAMPLING_TEMP]
                           [--out

In [37]:
#@title 4-1. PDB -> jsonl 변환 + ProteinMPNN 실행
import os
os.makedirs("mpnn_out", exist_ok=True)

get_ipython().system('python ProteinMPNN/helper_scripts/parse_multiple_chains.py \\\n  --input_path outputs \\\n  --output_path mpnn_out/parsed.jsonl')

get_ipython().system('python ProteinMPNN/protein_mpnn_run.py \\\n  --jsonl_path mpnn_out/parsed.jsonl \\\n  --out_folder mpnn_out \\\n  --num_seq_per_target 8 \\\n  --sampling_temp "0.1" \\\n  --seed 37 \\\n  --batch_size 1')

----------------------------------------
chain_id_jsonl is NOT loaded
----------------------------------------
fixed_positions_jsonl is NOT loaded
----------------------------------------
pssm_jsonl is NOT loaded
----------------------------------------
omit_AA_jsonl is NOT loaded
----------------------------------------
bias_AA_jsonl is NOT loaded
----------------------------------------
tied_positions_jsonl is NOT loaded
----------------------------------------
bias by residue dictionary is not loaded, or not provided
----------------------------------------
discarded {'bad_chars': 0, 'too_long': 0, 'bad_seq_length': 0}
----------------------------------------
Number of edges: 48
Training noise level: 0.2A
Generating sequences for: design_0
8 sequences of length 100 generated in 4.1454 seconds


## 5. 결과 정리

ProteinMPNN이 뽑아낸 서열들을 FASTA로 보기 좋게 출력하고, 결과 파일들을 압축해서 다운로드할 수 있게 준비합니다.

In [38]:
#@title 5-1. 설계된 서열 확인
get_ipython().system('ls mpnn_out/seqs')
fasta_path = get_ipython().getoutput('ls mpnn_out/seqs/*.fa')[0]
with open(fasta_path) as f:
    content = f.read()
print(content)

design_0.fa
>design_0, score=1.4446, global_score=1.4446, fixed_chains=[], designed_chains=['A'], model_name=v_48_020, git_hash=8907e6671bfbfc92303b5f79c4b5e6ce47cdef57, seed=37
GGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGGG
>T=0.1, sample=1, score=1.0203, global_score=1.0203, seq_recovery=0.0000
MEEKLKEALEKLKKEIEAAAKEEKVSKEEQEKLKEVLKEIEKVVEEAIKAAKENEEVLEKTLELLEKMIEEIKKNKKDVEKLIKELKELIKEIREIIEKA
>T=0.1, sample=2, score=1.0550, global_score=1.0550, seq_recovery=0.0000
MEEKLKKALEKLKKTIEKVAEEEKVSEEEKKKLEEVLKEIEEAIKEAIEAAKESEEILEKTLKLIEQMIKTIEENKKDIEKMVEKLKELIKELKEIIKKE
>T=0.1, sample=3, score=0.9523, global_score=0.9523, seq_recovery=0.0000
MEEEIKKALKELKEVVKKVIKEKNYSEEEKKKLEEVLKEVEKVAEEAKKAAEKNKEIKEATLKAFKEMIKVIKEEKDNVDKMVKKLKELIKKIKEEIKKA
>T=0.1, sample=4, score=0.9552, global_score=0.9552, seq_recovery=0.0000
MEEELKKALKELKKVAKEVIEKEKYSEEEKKKIEKALKEVEKVVKEAIEKAKKNEEIKKATLKALKEMIKVIKENKKDVEKMIKEIKKLIKEIKKTIKEA
>T=0.1, sample=5, score=0

In [39]:
#@title 5-2. 결과 압축 (다운로드용)
get_ipython().system('zip -qr protein_design_results.zip outputs mpnn_out')
get_ipython().system('ls -lh protein_design_results.zip')
from google.colab import files
files.download('protein_design_results.zip')

-rw-r--r-- 1 root root 463K Sep 21 07:00 protein_design_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>